# Notebook 02: Real Arm Baselines

**Purpose**: Run all 7 Family A demo-selection conditions on all real datasets, ID and OOD test sets, 2 models, 5 seeds.

**Compute budget**: 7 conditions x 5 seeds x 1000 rows x N datasets x 2 models LLM calls (~12-24h wall time on a single GPU).

## Why this notebook exists (thesis framing)

This notebook answers **RQ2**: *which kinds of demonstration diversity matter most for OOD performance?* It is also the primary evidence source for **RQ1** (alongside Notebook 01's pilot) and feeds the "best-performing protocol" selection used by Notebook 03's faithfulness evaluation and Notebook 05/06's SATA comparisons.

**Why compare 7 conditions rather than just testing SATA directly?** The lit review's Task 2 (Section 3) lays out four baselines specifically to isolate *which component* of the intervention is doing the work: (1) random demonstrations, to check whether demonstration design matters *at all*; (2) similarity-based retrieval (Liu et al. 2022, "What makes good in-context examples for GPT-3?"), to check whether the current best-practice retrieval method already solves OOD tabular tasks; (3) the four diversity protocols, to check whether principled, hand-designed selection closes the gap without any learning; and (4) SATA (Notebook 06), to check whether *learned, query-conditioned* reweighting adds anything beyond hand-designed diversity. This notebook covers baselines 1–3; SATA is evaluated on top of them in Notebook 06.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

## Conditions (7 total)

1. Zero-shot
2. Random-k
3. Similarity-k (all-MiniLM-L6-v2 cosine similarity)
4. Label diversity (stratified k/2 per class)
5. Feature-range diversity (quantile-bin coverage on top-3 continuous features)
6. Rule diversity (depth-3 decision tree leaf coverage)
7. Counter-spurious diversity (minority-cell over-sampling)

### Why these specific four diversity protocols, and why they're expected to differ

Two empirical findings from the lit review directly motivate this design:

- **Min et al. (2022), "Rethinking the role of demonstrations"** — across twelve LLMs including GPT-3, randomly corrupting demonstration *labels* barely hurts ICL accuracy. Ground-truth input–label mapping isn't the key signal; what matters is the **label space**, the **input-text distribution** the demonstrations sample, and the **input-label format** the demonstrations span. This is why demonstration design has room to operate on *which regions of input space and label space* the demos cover — that's exactly what protocols 4–7 manipulate — rather than needing to retrieve "more correct" examples.
- **Harutyunyan et al. (2024) and Chen et al. (2026)** — in-context learners are themselves susceptible to spurious correlations carried in the demonstration set, and this is a distinct failure mode from the base model's pretraining biases. Protocol 7 (counter-spurious) exists specifically to counteract this at the demonstration level, by denying the model the option of relying on the shortcut.

Each protocol targets a **different point in the covariate/concept shift taxonomy** (Lit-review §2.1.1), which is the basis for RQ2's success criterion:
- **Label diversity** guards against *label (prior-probability) shift* — a skewed class prior in the demo pool would bias the model's implicit prior even when `P(y|x)` hasn't moved.
- **Feature-range diversity** targets *covariate shift* (`P(x)` moves, `P(y|x)` stable) — forcing demos to span the full input range denies the model a narrow, spuriously-correlated cluster to anchor on.
- **Rule diversity** and **counter-spurious diversity** both target *concept shift* (`P(y|x)` itself moves) — the harder regime, per WHYSHIFT (Liu et al. 2023, Lit-review §2.1.3), because no amount of reweighting recovers a moved conditional without information about *how* the decision rule varies across regimes.

**RQ2 succeeds** only if at least one protocol beats random selection *and* different protocols win on different shift types — i.e. the interaction effect is the point, not any single protocol dominating everywhere. A protocol that's uniformly best (or uniformly no better than random) would actually be a weaker result for RQ2 than protocols that trade off differently across shift types.

In [2]:
from src.selection import (
    random_select,
    similarity_select,
    label_diversity,
    feature_range,
    rule_diversity,
    counter_spurious,
)

CONDITIONS = [
    'zero_shot', 'random', 'similarity', 'label_diversity',
    'feature_range', 'rule_diversity', 'counter_spurious',
]

## Prompt template & LLM runner

See `src/inference/prompts.py::build_classification_prompt` and `src/inference/llm_runner.py::VLLMRunner`.

SamplingParams: `logprobs=True, max_tokens=1, temperature=0`. Top token not in the label set -> log as `INVALID`, exclude from accuracy but include in the count.

**Why constrained single-token decoding instead of free-text generation?** This makes the prediction a clean forced choice between exactly the label tokens, so `logprob_0`/`logprob_1` are directly comparable across every condition/dataset/model combination and feed straight into the uncertainty analysis in Notebook 07 (R-AUC, F1@95%) — see `src/inference/llm_runner.py::get_confidence`'s constrained softmax. `temperature=0` makes predictions deterministic given the prompt, so any variation in results across seeds is attributable entirely to which demonstrations were selected, not to sampling noise in generation.

In [ ]:
import json

import numpy as np
import pandas as pd
from tqdm import tqdm

from src.data.tableshift_loader import SELECTED_DATASETS, load_tableshift_splits, select_top_features
from src.data.serialisation import serialise_row, ordered_feature_names
from src.inference.llm_runner import VLLMRunner
from src.inference.prompts import build_classification_prompt
from src.selection.rule_diversity import fit_leaf_tree
from src.selection.counter_spurious import find_spurious_proxy_features
from src.utils.results_schema import append_results, new_results_frame, load_results

# TableShift's domain_split_varname per dataset (tableshift/configs/benchmark_configs.py)
# — the variable that defines the OOD shift, used by the counter-spurious protocol
# to find features that proxy for it.
DOMAIN_SPLIT_VARNAME = {
    'acsincome': 'DIVISION',
    'acspubcov': 'DIS',
    'brfss_diabetes': 'PRACE1',
    'anes': 'VCF0112',
}

RESULTS_PATH = resolve_path('results/real_arm_baselines.parquet')


def prepare_condition_artifacts(dataset_name, train_pool, feature_cols):
    """One-time-per-dataset setup shared across seeds/queries for the
    feature_range, rule_diversity, and counter_spurious protocols.
    """
    artifacts = {}

    # Feature-range: top-3 continuous (numeric, >10 unique values) features by MI.
    continuous_cols = [
        c for c in feature_cols
        if pd.api.types.is_numeric_dtype(train_pool[c]) and train_pool[c].nunique() > 10
    ]
    if continuous_cols:
        artifacts['top3_continuous'] = select_top_features(
            train_pool[continuous_cols + ['label']], n_features=min(3, len(continuous_cols))
        )
    else:
        artifacts['top3_continuous'] = feature_cols[:3]

    # Rule diversity: depth-3 tree fit on the pool.
    artifacts['tree'] = fit_leaf_tree(train_pool, feature_cols)

    # Counter-spurious: proxy feature most correlated with both the label and
    # the shift variable. The shift variable isn't among the saved feature_cols
    # (it's usually not predictive enough to survive MI-based reduction), so we
    # reload the full unreduced training split (cached — near-instant) to get it.
    shift_col = DOMAIN_SPLIT_VARNAME[dataset_name]
    full_train = load_tableshift_splits(dataset_name)['train']
    shift_values = full_train[shift_col]
    if not pd.api.types.is_numeric_dtype(shift_values):
        shift_values = pd.Series(pd.factorize(shift_values)[0], index=full_train.index)
    proxy_frame = full_train[feature_cols + ['label']].copy()
    proxy_frame['_shift_code'] = shift_values
    proxy_features = find_spurious_proxy_features(proxy_frame, feature_cols, 'label', '_shift_code', top_n=3)
    proxy_col = proxy_features[0] if proxy_features else feature_cols[0]
    proxy_high = proxy_frame[proxy_col] > proxy_frame[proxy_col].median()
    artifacts['proxy_col'] = proxy_col
    artifacts['proxy_majority_label'] = proxy_frame.loc[proxy_high, 'label'].mode().iloc[0]

    return artifacts


def select_demos(condition, pool, query, k, seed, feature_cols, artifacts):
    if condition == 'zero_shot':
        return []
    if condition == 'random':
        return random_select.select(pool, query, k, seed)
    if condition == 'similarity':
        pool_texts = [
            serialise_row(
                {f: pool.loc[i, f] for f in ordered_feature_names({f: pool.loc[i, f] for f in feature_cols})},
                label=str(pool.loc[i, 'label']),
            )
            for i in pool.index
        ]
        query_text = serialise_row({f: query[f] for f in ordered_feature_names({f: query[f] for f in feature_cols})})
        local_idx = similarity_select.select(pool_texts, query_text, k, seed)
        return [pool.index[i] for i in local_idx]
    if condition == 'label_diversity':
        return label_diversity.select(pool, query, k, seed)
    if condition == 'feature_range':
        return feature_range.select(pool, query, k, seed, top_features=artifacts['top3_continuous'])
    if condition == 'rule_diversity':
        return rule_diversity.select(pool, query, k, seed, feature_cols=feature_cols, tree=artifacts['tree'])
    if condition == 'counter_spurious':
        return counter_spurious.select(
            pool, query, k, seed, proxy_col=artifacts['proxy_col'], proxy_majority_label=artifacts['proxy_majority_label']
        )
    raise ValueError(f"Unknown condition: {condition}")


def build_demo_lines(pool, demo_ids, feature_cols):
    lines = []
    for i in demo_ids:
        row = pool.loc[i]
        ordered = ordered_feature_names({f: row[f] for f in feature_cols})
        lines.append(serialise_row({f: row[f] for f in ordered}, label=str(row['label'])))
    return lines


def build_query_line(query, feature_cols):
    ordered = ordered_feature_names({f: query[f] for f in feature_cols})
    return serialise_row({f: query[f] for f in ordered})


try:
    import vllm  # noqa: F401
    VLLM_AVAILABLE = True
except ImportError:
    VLLM_AVAILABLE = False
    print("vLLM not installed in this environment — skipping real-arm inference. "
          "Run this notebook on a GPU box with vllm + the model weights available.")

# Model-outer / dataset-inner so each model's weights load exactly once
# (vLLM model load is the expensive step, not iterating datasets/conditions/seeds).
# Compute budget per the docstring above: 7 conditions x 5 seeds x 1000 rows x
# N datasets x 2 models (~12-24h wall time) — nested tqdm bars below give a
# glanceable readout of exactly which (model, dataset, condition, seed) is running.
models_to_run = config.base_llms if VLLM_AVAILABLE else []
model_bar = tqdm(models_to_run, desc="Models", position=0)
for model_cfg in model_bar:
    model_bar.set_postfix(model=model_cfg.name)
    runner = VLLMRunner(model_cfg.path, **vars(config.vllm))

    dataset_bar = tqdm(SELECTED_DATASETS, desc="Datasets", position=1, leave=False)
    for dataset_name in dataset_bar:
        dataset_bar.set_postfix(dataset=dataset_name)
        data_dir = resolve_path(config.paths.data_real) / dataset_name
        train_pool = pd.read_parquet(data_dir / 'train_pool.parquet')
        test_id = pd.read_parquet(data_dir / 'test_id.parquet')
        test_ood = pd.read_parquet(data_dir / 'test_ood.parquet')
        feature_cols = json.load(open(data_dir / 'feature_list.json'))
        label_tokens = tuple(json.load(open(data_dir / 'label_tokens.json')))
        task_description = f"the '{dataset_name}' outcome"

        artifacts = prepare_condition_artifacts(dataset_name, train_pool, feature_cols)

        condition_bar = tqdm(CONDITIONS, desc="Conditions", position=2, leave=False)
        for condition in condition_bar:
            condition_bar.set_postfix(condition=condition)
            seed_bar = tqdm(config.seed_accuracy, desc="Seeds", position=3, leave=False)
            for seed in seed_bar:
                seed_bar.set_postfix(seed=int(seed))
                k = config.k_primary
                batch_rows = []
                prompts = []
                for environment, test_df in [('id', test_id), ('ood', test_ood)]:
                    for query_id, (_, query) in enumerate(test_df.iterrows()):
                        demo_ids = select_demos(condition, train_pool, query, k, seed, feature_cols, artifacts)
                        demo_lines = build_demo_lines(train_pool, demo_ids, feature_cols)
                        query_line = build_query_line(query, feature_cols)
                        prompt = build_classification_prompt(task_description, label_tokens, demo_lines, query_line)
                        prompts.append(prompt)
                        batch_rows.append({
                            'arm': 'real', 'dataset': dataset_name, 'environment': environment,
                            'model': model_cfg.name, 'method': condition, 'seed': int(seed),
                            'query_id': query_id, 'label': str(query['label']),
                            'demo_ids': [int(i) for i in demo_ids], 'k': len(demo_ids),
                        })

                predictions = runner.batch_predict(prompts, label_tokens)
                for row, pred in zip(batch_rows, predictions):
                    row['prediction'] = pred.prediction
                    row['logprob_0'] = pred.logprob_0
                    row['logprob_1'] = pred.logprob_1

                append_results(pd.DataFrame(batch_rows), RESULTS_PATH)
                tqdm.write(f"{model_cfg.name} | {dataset_name} | {condition} | seed={seed}: {len(batch_rows)} rows")

## Output

- `results/real_arm_baselines.parquet`
- Summary table: accuracy, macro-F1, shift gap per (dataset, model, condition), averaged over seeds with std.

In [4]:
from src.evaluation.accuracy import summarise
from src.utils.results_schema import load_results

try:
    results = load_results(RESULTS_PATH)
except FileNotFoundError:
    results = None

if results is None or results.empty:
    print("No results yet — results/real_arm_baselines.parquet doesn't exist "
          "(the inference cell above was skipped since vLLM isn't available here).")
    summary = pd.DataFrame(columns=[
        'dataset', 'model', 'method', 'environment', 'accuracy_mean', 'accuracy_std',
        'macro_f1_mean', 'macro_f1_std', 'invalid_rate_mean', 'n', 'shift_gap',
    ])
else:
    # Per-seed accuracy/macro-F1 first, then mean +/- std *across seeds* (not a
    # single pooled accuracy over all seeds' rows).
    per_seed = summarise(results, group_cols=['dataset', 'model', 'method', 'environment', 'seed'])
    summary = per_seed.groupby(['dataset', 'model', 'method', 'environment']).agg(
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', 'std'),
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std'),
        invalid_rate_mean=('invalid_rate', 'mean'),
        n=('n', 'sum'),
    ).reset_index()

    shift_gap = summary.pivot_table(index=['dataset', 'model', 'method'], columns='environment', values='accuracy_mean')
    shift_gap = (shift_gap['id'] - shift_gap['ood']).rename('shift_gap').reset_index()
    summary = summary.merge(shift_gap, on=['dataset', 'model', 'method'], how='left')

# Saved separately from the schema'd per-prediction parquet — Notebook 03 reads
# this to pick the "best-performing protocol from Notebook 02" per (dataset, model).
summary.to_parquet(resolve_path('results/real_arm_baselines_summary.parquet'), index=False)
summary

No results yet — results/real_arm_baselines.parquet doesn't exist (the inference cell above was skipped since vLLM isn't available here).


,dataset,model,method,environment,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,invalid_rate_mean,n,shift_gap
